In [2]:
import requests
import json
import os

API example:

GET /api/intelligence/v3/vulnerabilities/production

Host: www.wordfence.com

Authorization: Bearer [YOUR_API_KEY]



In [ ]:
url = 'https://www.wordfence.com/api/intelligence/v3/vulnerabilities/production'
api = 'API_KEY_HERE'

headers = {
    'Authorization' : f'Bearer {api}'
}

#rsp = requests.get(url, headers=headers)
#rsp.json()

I think I've gotten all the data... Need to verify and clean.


In [ ]:
data_path = "../data/test.json"
with open(data_path, "r") as f:
    data = json.load(f)


So for the data it's the initial key is the uuid, values are all the info in the json 
We have roughly 36221 vulnerabilities in this database. 

Going to only focus on plug-ins 


Going to want to extract:
filter on ['software']['type'] = 'plugins'
['software']['name']
['software']['affected_versions']
['description']
['cvss']['score']
['cve']
That should cover pretty much all the data we need for a conversation. Use the CVE as the key and then populate it as a similiar 
id = cve
score = cvss score
type = software type
name = software name
versions = software affected versions
description = description 

Can feed this into an LLM 
Make sure to keep the raw data too in case we need to fix it. 

Going to create a ChatLM format too 

In [54]:
vuln_data = {}
for entry in data.values():
    if entry.get("cve"):
        if entry["software"][0]["type"] == "plugin":
            vuln_data[entry["cve"]] = {
                'slug' : entry['software'][0]['slug'],
                'description' : entry['description'],
                'score' : entry['cvss']['score'],
                'fix' : entry['software'][0]['remediation']
                }

In [55]:
write_data = json.dumps(vuln_data, indent=4)
with open("vuln_data.json", 'w') as f:
    f.write(write_data)

In [56]:
len(vuln_data)

30827